# Data prep

In [ ]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as f
from pyspark.sql.window import Window

from utils.consts import (
    SHARED_DIR_PATH,
    SPARK_CONFIGS
)

from utils.utils import SparkDataManager

sdm = SparkDataManager(SPARK_CONFIGS["FULL_RESOURCES"])

# cfg = SPARK_CONFIGS["HALF_SAFE"]

# spark = (
#     SparkSession.builder.appName("GenPM-fe")
#     .config(map=cfg)
#     .config("spark.log.level", "ERROR")
#     .getOrCreate()
# )

EDA_DATA_PATH = SHARED_DIR_PATH / "eda_data"
raw_pm_path = EDA_DATA_PATH / "raw_pm_data"
pm_kpi_pivot = EDA_DATA_PATH / "pm_data_pivot"
sample_path = EDA_DATA_PATH / "sample"
pm_stats_path = EDA_DATA_PATH / "stats"
pm_agg_path = EDA_DATA_PATH / "agg"
pm_metadata = EDA_DATA_PATH / "pm_metadata"

PREPROCESSED_DATASET_PATH = SHARED_DIR_PATH / "preprocessed_dataset"
long_path = PREPROCESSED_DATASET_PATH / "pm_data_long"
wide_path = PREPROCESSED_DATASET_PATH / "pm_data_wide"

FINAL_PREPROCESSED_DATASET_PATH = PREPROCESSED_DATASET_PATH / "final"
final_kpi_defs_path = FINAL_PREPROCESSED_DATASET_PATH / "kpi_definitions"
final_pm_data_dataset_path = FINAL_PREPROCESSED_DATASET_PATH / "pm_data_dataset_preprocessed"
pm_df_long_idx_win_path = FINAL_PREPROCESSED_DATASET_PATH / "pm_df_long_indexed_winds"
final_simple_reports_path = FINAL_PREPROCESSED_DATASET_PATH / "simple_reports"

TRAINING_DATASET_PATH = SHARED_DIR_PATH / "training_data"
windows_anchors_path = TRAINING_DATASET_PATH / "kpi_window_anchors"
windows_long_path = TRAINING_DATASET_PATH / "kpi_windows_long"

In [ ]:
# df_final = spark.read.parquet(str(final_pm_data_dataset_path / "_temporary/0/_temporary"))
df_long_idx = spark.read.parquet(str(pm_df_long_idx_win_path))
simple_reports = spark.read.parquet(str(final_simple_reports_path))
kpi_defs = spark.read.parquet(str(final_kpi_defs_path))

# df_final.show()
df_long_idx.show(400)
simple_reports.show()
kpi_defs.show()

In [ ]:
def validate_counts(df, seq_len):
    from pyspark.sql import functions as F

    base = df.select("distname", "bts_id", "kpi_id", "window_anchor", "hour_idx", "kpi_value", "imputed_flag")

    total_rows = base.count()
    n_distname = base.select("distname").distinct().count()
    n_bts = base.select("bts_id").distinct().count()
    n_kpi = base.select("kpi_id").distinct().count()
    n_windows = base.select("distname", "window_anchor").distinct().count()

    hours_per_window = (
        base.groupBy("distname", "window_anchor")
        .agg(
            F.countDistinct("hour_idx").alias("distinct_hours"),
            F.min("hour_idx").alias("min_hour_idx"),
            F.max("hour_idx").alias("max_hour_idx"),
        )
    )

    kpis_per_window_hour = (
        base.groupBy("distname", "window_anchor", "hour_idx")
        .agg(F.countDistinct("kpi_id").alias("distinct_kpis"))
    )

    windows_per_dist = base.select("distname", "window_anchor").distinct().groupBy("distname").count()

    validation = {
        "total_rows": total_rows,
        "unique_distname": n_distname,
        "unique_bts_id": n_bts,
        "unique_kpi_id": n_kpi,
        "unique_windows": n_windows,
        "rows_per_window_expected": seq_len * n_kpi,
        "estimated_rows_from_windows": int(n_windows * seq_len * n_kpi),
        "windows_per_distname_summary": windows_per_dist.agg(
            F.min("count").alias("min"),
            F.expr("percentile_approx(count, 0.5)").alias("median"),
            F.avg("count").alias("mean"),
            F.max("count").alias("max"),
        ).first().asDict(),
        "hours_per_window_summary": hours_per_window.agg(
            F.min("distinct_hours").alias("min"),
            F.expr("percentile_approx(distinct_hours, 0.5)").alias("median"),
            F.avg("distinct_hours").alias("mean"),
            F.max("distinct_hours").alias("max"),
        ).first().asDict(),
        "kpis_per_window_hour_summary": kpis_per_window_hour.agg(
            F.min("distinct_kpis").alias("min"),
            F.expr("percentile_approx(distinct_kpis, 0.5)").alias("median"),
            F.avg("distinct_kpis").alias("mean"),
            F.max("distinct_kpis").alias("max"),
        ).first().asDict(),
    }

    bad_windows_hours = hours_per_window.filter(F.col("distinct_hours") != seq_len).count()
    bad_window_hours_kpis = kpis_per_window_hour.filter(F.col("distinct_kpis") != n_kpi).count()
    validation["windows_with_nonstandard_hour_count"] = bad_windows_hours
    validation["window_hours_with_nonstandard_kpi_count"] = bad_window_hours_kpis

    return validation, windows_per_dist

In [ ]:
validation, windows_per_dist = validate_counts(df_long_idx, 168)
validation.show()
windows_per_dist.show()

In [ ]:
validation

In [ ]:
wide = (df_long_idx.groupBy("distname","bts_id","window_anchor","hour_idx")
        .pivot("kpi_id")
        .agg(f.first("kpi_value"))
        .orderBy("distname","window_anchor","hour_idx")).cache()
print(wide.count())
wide.show()

In [ ]:
sdm.write_parquet(
            wide, TRAINING_DATASET_PATH / "greg_tmp" / "wide_win_idx", mode="overwrite"
        )

# Modelling

In [ ]:
import numpy as np
import pandas as pd
from utils.consts import (
    SHARED_DIR_PATH,
    SPARK_CONFIGS
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import tsgm
import torch
import keras
import functools

TRAINING_DATASET_PATH = SHARED_DIR_PATH / "training_data"

In [ ]:
pdf = pd.read_parquet(TRAINING_DATASET_PATH / "greg_tmp" / "wide_win_idx")

In [ ]:
feat_cols = sorted([c for c in pdf.columns if c not in ["distname","bts_id","window_anchor","hour_idx"]])

X = np.stack([
    g.sort_values("hour_idx")[feat_cols].to_numpy(dtype=np.float32)
    for (_, _), g in pdf.groupby(["distname","window_anchor"], sort=False)
    if len(g) == 168 and not np.isnan(g[feat_cols].to_numpy()).any()
]).astype(np.float32)

scaler = tsgm.utils.TSFeatureWiseScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

In [ ]:
LABEL_COL = "distname"

In [ ]:
y = np.array([
    g[LABEL_COL].iloc[0]
    for (_, _), g in pdf.groupby(["distname", "window_anchor"], sort=False)
    if len(g) == 168 and not np.isnan(g[feat_cols].to_numpy()).any()
])
le = LabelEncoder()
y_int = le.fit_transform(y)
y_onehot = keras.utils.to_categorical(y_int).astype(np.float32)

In [ ]:
y_onehot.shape

In [ ]:
# keras.mixed_precision.set_global_policy("mixed_float16")


In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
x = torch.randn(2, 2).to("cuda")
print(x.device)

In [ ]:
seq_len = X_scaled.shape[1]
feat_dim = X_scaled.shape[2]
n_classes = y_onehot.shape[1]

In [ ]:
from data_loading_pipe import to_numpy, unscale, discriminative_score

In [ ]:
import numpy as np
import pandas as pd
from utils.consts import (
    SHARED_DIR_PATH,
    SPARK_CONFIGS
)
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
import tsgm
import torch
import keras
import functools

from model_utils import cVAE_LSTMv3Architecture, cVAE_LSTMv2Architecture


TRAINING_DATASET_PATH = SHARED_DIR_PATH / "training_data"

pdf = pd.read_parquet(TRAINING_DATASET_PATH / "greg_tmp" / "wide_win_idx")

feat_cols = sorted([c for c in pdf.columns if c not in ["distname","bts_id","window_anchor","hour_idx"]])

X = np.stack([
    g.sort_values("hour_idx")[feat_cols].to_numpy(dtype=np.float32)
    for (_, _), g in pdf.groupby(["distname","window_anchor"], sort=False)
    if len(g) == 168 and not np.isnan(g[feat_cols].to_numpy()).any()
]).astype(np.float32)

scaler = tsgm.utils.TSFeatureWiseScaler()
X_scaled = scaler.fit_transform(X).astype(np.float32)

LABEL_COL = "distname"

y = np.array([
    g[LABEL_COL].iloc[0]
    for (_, _), g in pdf.groupby(["distname", "window_anchor"], sort=False)
    if len(g) == 168 and not np.isnan(g[feat_cols].to_numpy()).any()
])
le = LabelEncoder()
y_int = le.fit_transform(y)
y_onehot = keras.utils.to_categorical(y_int).astype(np.float32)

seq_len = X_scaled.shape[1]
feat_dim = X_scaled.shape[2]
n_classes = y_onehot.shape[1]

LATENT_DIM    = 32    # was 64 — halves latent space; at 64 ~99% of dims were collapsed
HIDDEN_DIM    = 256   # keep — matched to feat_dim=239
N_LAYERS      = 2     # was 3 — saves ~33% per epoch, shorter gradient path
BATCH_SIZE    = 64    # keep
EPOCHS        = 150
ANNEAL_EPOCHS = 20
TARGET_BETA   = 1.0
arch = cVAE_LSTMv2Architecture(
    seq_len, feat_dim, LATENT_DIM, n_classes,
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    use_attention=True,   # temporal self-attention for 24h/7-day patterns
    n_heads=4,
)
model_cwel = tsgm.models.cvae.cBetaVAE(
    arch.encoder, arch.decoder, LATENT_DIM,
    temporal=False,
    beta=0.0,
)
model_cwel.compile(
    optimizer=keras.optimizers.Adam(5e-4, clipnorm=1.0),
)
class KLAnneal(keras.callbacks.Callback):
    def __init__(self, target, warmup):
        self.target, self.warmup = target, warmup
    def on_epoch_begin(self, epoch, logs=None):
        self.model.beta = self.target * min(1.0, (epoch + 1) / self.warmup)


model_cwel.fit(
    X_scaled, y_onehot,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[
        KLAnneal(TARGET_BETA, ANNEAL_EPOCHS),
        keras.callbacks.ReduceLROnPlateau(
            monitor="reconstruction_loss", mode="min",
            factor=0.5, patience=15, min_lr=1e-5,
        ),
        keras.callbacks.ModelCheckpoint(
            TRAINING_DATASET_PATH / "greg_tmp" / "cvel_lstm_v3.weights.h5",
            monitor="reconstruction_loss", mode="min",
            save_best_only=True, save_weights_only=True,
        ),
        keras.callbacks.EarlyStopping(
            monitor="reconstruction_loss", mode="min",
            patience=30, restore_best_weights=True,
        ),
    ],
    verbose=1,
)

## VAE

In [ ]:
arch = tsgm.models.zoo["vae_conv5"](seq_len, feat_dim, 64)
model_vae = tsgm.models.cvae.BetaVAE(arch.encoder, arch.decoder)

model_vae.compile(optimizer=keras.optimizers.Adam())
model_vae.fit(X_scaled, epochs=500, batch_size=32)

In [ ]:
arch = tsgm.models.zoo["vae_conv5"](X_scaled.shape[1], X_scaled.shape[2], 32)
model = tsgm.models.cvae.BetaVAE(arch.encoder, arch.decoder)

model.compile(optimizer=keras.optimizers.Adam())
model.fit(X_scaled, epochs=150, batch_size=16)

In [ ]:
arch = tsgm.models.zoo["vae_conv5"](X_scaled.shape[1], X_scaled.shape[2], 32)
model = tsgm.models.cvae.BetaVAE(arch.encoder, arch.decoder)

model.compile(optimizer=keras.optimizers.Adam())
model.fit(X_scaled, epochs=100, batch_size=32)

In [ ]:
arch = tsgm.models.zoo["vae_conv5"](X_scaled.shape[1], X_scaled.shape[2], 64)
model = tsgm.models.cvae.BetaVAE(arch.encoder, arch.decoder)

model.compile(optimizer=keras.optimizers.Adam())
model.fit(X_scaled, epochs=200, batch_size=168)

In [ ]:
# %% generate & score
X_vae_scaled = to_numpy(model_vae.generate(500))
X_vae = unscale(scaler, X_vae_scaled)
score = discriminative_score(X_scaled, X_vae_scaled, seq_len, feat_dim)
print(f"X_gen {X_vae.shape}  |  discriminative score {score:.3f}  (0.5 = ideal)")

In [ ]:
# %% generate & score
X_vae_scaled = to_numpy(model.generate(500))
X_vae = unscale(scaler, X_vae_scaled)
score = discriminative_score(X_scaled, X_vae_scaled, seq_len, feat_dim)
print(f"X_gen {X_vae.shape}  |  discriminative score {score:.3f}  (0.5 = ideal)")

In [ ]:
import pickle

In [ ]:
path = TRAINING_DATASET_PATH / "greg_tmp" / "vae_v0.weights.h5"

In [ ]:
model.save_weights(path)

## CVAE

In [ ]:
LATENT_DIM, BATCH_SIZE, EPOCHS = 64, 32, 300

arch = tsgm.models.zoo["cvae_conv5"](seq_len, feat_dim, LATENT_DIM, n_classes)
model_cvae = tsgm.models.cvae.cBetaVAE(arch.encoder, arch.decoder, LATENT_DIM, temporal=False, beta=1.0)
model_cvae.compile(optimizer=keras.optimizers.Adam(1e-3))

# %% train
model_cvae.fit(X_scaled, y_onehot, epochs=150, batch_size=32)

In [ ]:
path2 = TRAINING_DATASET_PATH / "greg_tmp" / "cvae_v0.weights.h5"
model_cvae.save_weights(path2)

In [ ]:
LATENT_DIM, BATCH_SIZE, EPOCHS = 32, 16, 150

arch = tsgm.models.zoo["cvae_conv5"](seq_len, feat_dim, LATENT_DIM, n_classes)
model_cvae2 = tsgm.models.cvae.cBetaVAE(arch.encoder, arch.decoder, LATENT_DIM, temporal=False, beta=3.0)
model_cvae2.compile(optimizer=keras.optimizers.Adam(1e-3))

# %% train
model_cvae2.fit(X_scaled, y_onehot, epochs=EPOCHS, batch_size=BATCH_SIZE)

In [ ]:
path3 = TRAINING_DATASET_PATH / "greg_tmp" / "cvae_v1.weights.h5"
model_cvae2.save_weights(path3)

## CVAEL

In [ ]:
import abc
import typing as T

In [ ]:
class Architecture(abc.ABC):
    @abc.abstractproperty
    def arch_type(self):
        raise NotImplementedError

In [ ]:
import keras
from keras import layers
from keras import ops
import math

In [ ]:
class Sampling(keras.layers.Layer):
    """
    Custom Keras layer for sampling from a latent space.

    This layer samples from a latent space using the reparameterization trick during training.
    It takes as input the mean and log variance of the latent distribution and generates
    samples by adding random noise scaled by the standard deviation to the mean.
    """
    def call(self, inputs: T.Tuple[tsgm.types.Tensor, tsgm.types.Tensor]) -> tsgm.types.Tensor:
        """
        Generates samples from a latent space.

        :param inputs: Tuple containing mean and log variance tensors of the latent distribution.
        :type inputs: tuple[tsgm.types.Tensor, tsgm.types.Tensor]

        :returns: Sampled latent vector.
        :rtype: tsgm.types.Tensor
        """
        z_mean, z_log_var = inputs
        #  random noise for keras3.0
        epsilon = keras.random.normal(shape=ops.shape(z_mean))
        #  ops for keras3.0
        return z_mean + ops.exp(0.5 * z_log_var) * epsilon

class BaseVAEArchitecture(Architecture):
    """
    Base class for defining architectures of Variational Autoencoders (VAEs).
    """
    @property
    def encoder(self) -> keras.models.Model:
        """
        Property for accessing the encoder model.

        :return: The encoder model.
        :rtype: keras.models.Model
        :raises NotImplementedError: If the encoder model is not implemented.
        """
        if hasattr(self, "_encoder"):
            return self._encoder
        else:
            raise NotImplementedError

    @property
    def decoder(self) -> keras.models.Model:
        """
        Property for accessing the decoder model.

        :return: The decoder model.
        :rtype: keras.models.Model
        :raises NotImplementedError: If the decoder model is not implemented.
        """
        if hasattr(self, "_decoder"):
            return self._decoder
        else:
            raise NotImplementedError

    def get(self) -> T.Dict:
        """
        Retrieves both encoder and decoder models as a dictionary.

        :return: A dictionary containing encoder and decoder models.
        :rtype: dict
        :raises NotImplementedError: If either encoder or decoder models are not implemented.
        """
        if hasattr(self, "_encoder") and hasattr(self, "_decoder"):
            return {"encoder": self._encoder, "decoder": self._decoder}
        else:
            raise NotImplementedError

In [ ]:
class VAE_LSTMArchitecture(BaseVAEArchitecture):
    """
    Variational Autoencoder with bidirectional LSTM encoder and LSTM decoder.

    Better suited than Conv1D for long sequences (e.g. 168-hour telecom KPI windows)
    where temporal dependencies span the full sequence length.
    """
    arch_type = "vae:unconditional"

    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        latent_dim: int,
        hidden_dim: int = 128,
        n_layers: int = 2,
    ) -> None:
        super().__init__()
        self._seq_len = seq_len
        self._feat_dim = feat_dim
        self._latent_dim = latent_dim
        self._hidden_dim = hidden_dim
        self._n_layers = n_layers
        self._encoder = self._build_encoder()
        self._decoder = self._build_decoder()

    def _build_encoder(self) -> keras.models.Model:
        encoder_inputs = keras.Input(shape=(self._seq_len, self._feat_dim))
        x = encoder_inputs
        for i in range(self._n_layers - 1):
            x = layers.Bidirectional(
                layers.LSTM(self._hidden_dim, return_sequences=True)
            )(x)
            x = layers.Dropout(rate=0.2)(x)
        x = layers.Bidirectional(
            layers.LSTM(self._hidden_dim, return_sequences=False)
        )(x)
        z_mean = layers.Dense(self._latent_dim, name="z_mean")(x)
        z_log_var = layers.Dense(self._latent_dim, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    def _build_decoder(self) -> keras.models.Model:
        latent_inputs = keras.Input(shape=(self._latent_dim,))
        x = layers.RepeatVector(self._seq_len)(latent_inputs)
        for _ in range(self._n_layers):
            x = layers.LSTM(self._hidden_dim, return_sequences=True)(x)
            x = layers.Dropout(rate=0.2)(x)
        decoder_outputs = layers.TimeDistributed(
            layers.Dense(self._feat_dim, activation="sigmoid")
        )(x)
        return keras.Model(latent_inputs, decoder_outputs, name="decoder")


class cVAE_LSTMArchitecture(BaseVAEArchitecture):
    """
    Conditional VAE with bidirectional LSTM encoder and LSTM decoder.

    Compatible with cBetaVAE. Per-timestep latent of size latent_dim;
    total latent vector size is latent_dim * seq_len.
    """
    arch_type = "vae:conditional"

    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        latent_dim: int,
        output_dim: int = 2,
        hidden_dim: int = 128,
        n_layers: int = 2,
    ) -> None:
        self._seq_len = seq_len
        self._feat_dim = feat_dim
        self._latent_dim = latent_dim
        self._output_dim = output_dim
        self._hidden_dim = hidden_dim
        self._n_layers = n_layers
        self._encoder = self._build_encoder()
        self._decoder = self._build_decoder()

    def _build_encoder(self) -> keras.models.Model:
        encoder_inputs = keras.Input(
            shape=(self._seq_len, self._feat_dim + self._output_dim)
        )
        x = encoder_inputs
        for _ in range(self._n_layers):
            x = layers.Bidirectional(
                layers.LSTM(self._hidden_dim, return_sequences=True)
            )(x)
            x = layers.Dropout(rate=0.2)(x)
        x = layers.TimeDistributed(
            layers.Dense(self._latent_dim, activation="relu")
        )(x)
        x = layers.Flatten()(x)
        latent_size = self._latent_dim * self._seq_len
        z_mean = layers.Dense(latent_size, name="z_mean")(x)
        z_log_var = layers.Dense(latent_size, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    def _build_decoder(self) -> keras.models.Model:
        inputs = keras.Input(
            shape=(self._seq_len, self._latent_dim + self._output_dim)
        )
        x = inputs
        for _ in range(self._n_layers):
            x = layers.LSTM(self._hidden_dim, return_sequences=True)(x)
            x = layers.Dropout(rate=0.2)(x)
        d_output = layers.TimeDistributed(
            layers.Dense(self._feat_dim, activation="sigmoid")
        )(x)
        return keras.Model(inputs, d_output, name="decoder")


In [ ]:
class cVAE_LSTMv2Architecture(BaseVAEArchitecture):
    """
    Improved conditional VAE with bidirectional LSTM encoder and LSTM decoder.

    Enhancements over cVAE_LSTMArchitecture:
    - LayerNormalization after each BiLSTM/LSTM layer (stabilises gradients,
      eliminates loss spikes seen with deep stacked LSTMs on high-dim PM data).
    - Optional temporal self-attention block in the encoder after the LSTM stack
      (captures long-range daily/weekly periodicity in hourly PM windows such as
      seq_len=168 = one week at 1-h granularity).

    Recommended defaults for a 239-KPI, 168-step, multi-class telecom dataset:
        latent_dim=32, hidden_dim=256, n_layers=2, use_attention=True
    """
    arch_type = "vae:conditional_v2"

    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        latent_dim: int,
        output_dim: int = 2,
        hidden_dim: int = 256,
        n_layers: int = 2,
        use_attention: bool = True,
        n_heads: int = 4,
    ) -> None:
        self._seq_len = seq_len
        self._feat_dim = feat_dim
        self._latent_dim = latent_dim
        self._output_dim = output_dim
        self._hidden_dim = hidden_dim
        self._n_layers = n_layers
        self._use_attention = use_attention
        self._n_heads = n_heads
        self._encoder = self._build_encoder()
        self._decoder = self._build_decoder()

    def _build_encoder(self) -> keras.models.Model:
        encoder_inputs = keras.Input(
            shape=(self._seq_len, self._feat_dim + self._output_dim)
        )
        x = encoder_inputs
        for _ in range(self._n_layers):
            x = layers.Bidirectional(
                layers.LSTM(self._hidden_dim, return_sequences=True)
            )(x)
            x = layers.LayerNormalization()(x)
            x = layers.Dropout(rate=0.1)(x)

        if self._use_attention:
            # key_dim = hidden per head; use bidirectional output width = 2 * hidden_dim
            attn_out = layers.MultiHeadAttention(
                num_heads=self._n_heads,
                key_dim=self._hidden_dim // self._n_heads,
                dropout=0.1,
            )(x, x)
            x = layers.Add()([x, attn_out])
            x = layers.LayerNormalization()(x)

        x = layers.TimeDistributed(
            layers.Dense(self._latent_dim, activation="relu")
        )(x)
        x = layers.Flatten()(x)
        latent_size = self._latent_dim * self._seq_len
        z_mean = layers.Dense(latent_size, name="z_mean")(x)
        z_log_var = layers.Dense(latent_size, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    def _build_decoder(self) -> keras.models.Model:
        inputs = keras.Input(
            shape=(self._seq_len, self._latent_dim + self._output_dim)
        )
        x = inputs
        for _ in range(self._n_layers):
            x = layers.LSTM(self._hidden_dim, return_sequences=True)(x)
            x = layers.LayerNormalization()(x)
            x = layers.Dropout(rate=0.1)(x)
        d_output = layers.TimeDistributed(
            layers.Dense(self._feat_dim, activation="sigmoid")
        )(x)
        return keras.Model(inputs, d_output, name="decoder")


In [ ]:
class HourlyPositionalEncoding(keras.layers.Layer):
    """
    Sinusoidal positional encoding tuned to hourly telecom PM windows.

    Concatenates 4 fixed channels to the last dimension of the input:
        sin(2π·t/24),  cos(2π·t/24)   — hour-of-day cycle (period = 24 h)
        sin(2π·t/168), cos(2π·t/168)  — day-of-week cycle (period = 168 h)

    This gives the LSTM and attention head explicit knowledge of where each
    timestep sits within the daily and weekly cycle, which is the dominant
    structure in cellular network KPIs.  Works regardless of the window stride
    used to build training samples (e.g. stride=24 h).

    Input shape:  (batch, seq_len, features)
    Output shape: (batch, seq_len, features + 4)
    """

    def __init__(self, seq_len: int, **kwargs) -> None:
        super().__init__(**kwargs)
        self._seq_len = seq_len
        t = ops.arange(seq_len, dtype="float32")
        sin_24  = ops.sin(2.0 * math.pi * t / 24.0)
        cos_24  = ops.cos(2.0 * math.pi * t / 24.0)
        sin_168 = ops.sin(2.0 * math.pi * t / 168.0)
        cos_168 = ops.cos(2.0 * math.pi * t / 168.0)
        # shape: (seq_len, 4) — constant, no trainable weights
        self._pe = ops.stack([sin_24, cos_24, sin_168, cos_168], axis=-1)

    def call(self, x: tsgm.types.Tensor) -> tsgm.types.Tensor:
        batch = ops.shape(x)[0]
        pe = ops.broadcast_to(self._pe[None, :, :], (batch, self._seq_len, 4))
        return ops.concatenate([x, pe], axis=-1)

    def get_config(self) -> T.Dict:
        return {**super().get_config(), "seq_len": self._seq_len}

In [ ]:
class cVAE_LSTMv3Architecture(BaseVAEArchitecture):
    """
    Improved conditional VAE with bidirectional LSTM encoder and LSTM decoder.

    Enhancements over cVAE_LSTMArchitecture:
    - HourlyPositionalEncoding prepended to both encoder and decoder inputs.
      Injects sin/cos of the 24-h daily cycle and 168-h weekly cycle so that
      the LSTM and attention layer know the hour-of-day position of each step.
      Critical when windows are built with stride < seq_len (e.g. stride=24 on
      168-h windows) because the daily pattern repeats 7× inside each window.
    - LayerNormalization after each BiLSTM/LSTM layer (eliminates the gradient
      magnitude spikes observed in deep LSTM stacks on high-dim PM data).
    - Optional temporal self-attention block in the encoder after the LSTM stack
      to directly relate step t to step t+24 (same hour yesterday) and t+48, etc.

    Recommended defaults for a 239-KPI, 168-step, multi-class telecom dataset
    with stride=24 windowing:
        latent_dim=32, hidden_dim=256, n_layers=2, use_attention=True, n_heads=4
    """
    arch_type = "vae:conditional_v2"

    def __init__(
        self,
        seq_len: int,
        feat_dim: int,
        latent_dim: int,
        output_dim: int = 2,
        hidden_dim: int = 256,
        n_layers: int = 2,
        use_attention: bool = True,
        n_heads: int = 4,
    ) -> None:
        self._seq_len = seq_len
        self._feat_dim = feat_dim
        self._latent_dim = latent_dim
        self._output_dim = output_dim
        self._hidden_dim = hidden_dim
        self._n_layers = n_layers
        self._use_attention = use_attention
        self._n_heads = n_heads
        self._encoder = self._build_encoder()
        self._decoder = self._build_decoder()

    def _build_encoder(self) -> keras.models.Model:
        encoder_inputs = keras.Input(
            shape=(self._seq_len, self._feat_dim + self._output_dim)
        )
        # Inject hour-of-day and day-of-week position features before the LSTM.
        # Output shape: (batch, seq_len, feat_dim + output_dim + 4)
        x = HourlyPositionalEncoding(self._seq_len)(encoder_inputs)

        for _ in range(self._n_layers):
            x = layers.Bidirectional(
                layers.LSTM(self._hidden_dim, return_sequences=True)
            )(x)
            x = layers.LayerNormalization()(x)
            x = layers.Dropout(rate=0.1)(x)

        if self._use_attention:
            # BiLSTM output width = 2 * hidden_dim; key_dim splits across heads.
            attn_out = layers.MultiHeadAttention(
                num_heads=self._n_heads,
                key_dim=(2 * self._hidden_dim) // self._n_heads,
                dropout=0.1,
            )(x, x)
            x = layers.Add()([x, attn_out])
            x = layers.LayerNormalization()(x)

        x = layers.TimeDistributed(
            layers.Dense(self._latent_dim, activation="relu")
        )(x)
        x = layers.Flatten()(x)
        latent_size = self._latent_dim * self._seq_len
        z_mean = layers.Dense(latent_size, name="z_mean")(x)
        z_log_var = layers.Dense(latent_size, name="z_log_var")(x)
        z = Sampling()([z_mean, z_log_var])
        return keras.Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

    def _build_decoder(self) -> keras.models.Model:
        inputs = keras.Input(
            shape=(self._seq_len, self._latent_dim + self._output_dim)
        )
        # Decoder also gets positional context so it knows which hour to generate.
        x = HourlyPositionalEncoding(self._seq_len)(inputs)

        for _ in range(self._n_layers):
            x = layers.LSTM(self._hidden_dim, return_sequences=True)(x)
            x = layers.LayerNormalization()(x)
            x = layers.Dropout(rate=0.1)(x)
        d_output = layers.TimeDistributed(
            layers.Dense(self._feat_dim, activation="sigmoid")
        )(x)
        return keras.Model(inputs, d_output, name="decoder")

In [ ]:
keras.mixed_precision.set_global_policy("mixed_float16")
LATENT_DIM    = 32    # was 64 — halves latent space; at 64 ~99% of dims were collapsed
HIDDEN_DIM    = 256   # keep — matched to feat_dim=239
N_LAYERS      = 2     # was 3 — saves ~33% per epoch, shorter gradient path
BATCH_SIZE    = 64    # keep
EPOCHS        = 300
ANNEAL_EPOCHS = 20
TARGET_BETA   = 1.0
arch = cVAE_LSTMv2Architecture(
    seq_len, feat_dim, LATENT_DIM, n_classes,
    hidden_dim=HIDDEN_DIM,
    n_layers=N_LAYERS,
    use_attention=True,   # temporal self-attention for 24h/7-day patterns
    n_heads=4,
)
model_cwel = tsgm.models.cvae.cBetaVAE(
    arch.encoder, arch.decoder, LATENT_DIM,
    temporal=False,
    beta=0.0,
)
model_cwel.compile(
    optimizer=keras.optimizers.Adam(5e-4, clipnorm=1.0),
)
class KLAnneal(keras.callbacks.Callback):
    def __init__(self, target, warmup):
        self.target, self.warmup = target, warmup
    def on_epoch_begin(self, epoch, logs=None):
        self.model.beta = self.target * min(1.0, (epoch + 1) / self.warmup)


model_cwel.fit(
    X_scaled, y_onehot,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[
        KLAnneal(TARGET_BETA, ANNEAL_EPOCHS),
        keras.callbacks.ReduceLROnPlateau(
            monitor="reconstruction_loss", mode="min",
            factor=0.5, patience=15, min_lr=1e-5,
        ),
        keras.callbacks.ModelCheckpoint(
            TRAINING_DATASET_PATH / "greg_tmp" / "cvel_lstm_v3.weights.h5",
            monitor="reconstruction_loss", mode="min",
            save_best_only=True, save_weights_only=True,
        ),
        keras.callbacks.EarlyStopping(
            monitor="reconstruction_loss", mode="min",
            patience=30, restore_best_weights=True,
        ),
    ],
    verbose=1,
)

In [ ]:
LATENT_DIM, HIDDEN_DIM, N_LAYERS = 16, 128, 3
BATCH_SIZE, EPOCHS = 128, 200

arch = cVAE_LSTMArchitecture(seq_len, feat_dim, LATENT_DIM, n_classes, HIDDEN_DIM, N_LAYERS)
model_cwel = tsgm.models.cvae.cBetaVAE(arch.encoder, arch.decoder, LATENT_DIM, temporal=False, beta=2.0)
model_cwel.compile(optimizer=keras.optimizers.Adam(2e-3))

# %% train
model_cwel.fit(X_scaled, y_onehot, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)

In [ ]:
LATENT_DIM  = 64      # was 16  — gives 64×168 = 10,752 latent dims, enough capacity
HIDDEN_DIM  = 256     # was 128 — match the high feat_dim of 239
N_LAYERS    = 3       # keep
BATCH_SIZE  = 64      # was 128 — smaller batches help LSTM gradient quality
EPOCHS      = 300     # was 200 — more budget since LR is lower
arch = cVAE_LSTMArchitecture(seq_len, feat_dim, LATENT_DIM, n_classes, HIDDEN_DIM, N_LAYERS)
model_cwel = tsgm.models.cvae.cBetaVAE(
    arch.encoder, arch.decoder, LATENT_DIM,
    temporal=False,
    beta=1.0,          # was 2.0 — reduce KL pressure until reconstruction converges
)
model_cwel.compile(
    optimizer=keras.optimizers.Adam(5e-4),   # was 2e-3
)
lr_cb = keras.callbacks.ReduceLROnPlateau(
    monitor="reconstruction_loss",
    mode='min',
    factor=0.5,
    patience=15,
    min_lr=1e-5
)
model_cwel.fit(
    X_scaled, y_onehot,
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[lr_cb], verbose=1,
)

In [ ]:
cvae_labels = y_onehot
X_cvae_scaled, _ = model_cvae2.generate(cvae_labels)
X_cvae_scaled = to_numpy(X_cvae_scaled)
X_cvae = unscale(scaler, X_cvae_scaled)
score = discriminative_score(X_scaled, X_cvae_scaled, seq_len, feat_dim)
print(f"X_gen {X_cvae.shape}  |  discriminative score {score:.3f}  (0.5 = ideal)")

## GAN

In [ ]:
LATENT_DIM, BATCH_SIZE, EPOCHS = 128, 128, 1000

arch = tsgm.models.zoo["cgan_base_c4_l1"](seq_len, feat_dim, LATENT_DIM, output_dim=0)
model_gan = tsgm.models.cgan.GAN(arch.discriminator, arch.generator, LATENT_DIM, use_wgan=False)
model_gan.compile(
    d_optimizer=keras.optimizers.Adam(1e-4, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(1e-4, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=True),
)

# %% train
model_gan.fit(X_scaled, epochs=150, batch_size=BATCH_SIZE, verbose= 1)

In [ ]:
path3 = TRAINING_DATASET_PATH / "greg_tmp" / "gan_v0.weights.h5"
model_gan.save_weights(path3)

## CGAN

In [ ]:
LATENT_DIM, BATCH_SIZE, EPOCHS = 16, 64, 10
arch = tsgm.models.zoo["cgan_base_c4_l1"](seq_len, feat_dim, LATENT_DIM, n_classes)
model = tsgm.models.cgan.ConditionalGAN(arch.discriminator, arch.generator, LATENT_DIM, temporal=False)
model.compile(
    d_optimizer=keras.optimizers.Adam(1e-4, beta_1=0.5),
    g_optimizer=keras.optimizers.Adam(1e-4, beta_1=0.5),
    loss_fn=keras.losses.BinaryCrossentropy(from_logits=True),
)

# %% train
model.fit(X_scaled, y_onehot, epochs=EPOCHS, batch_size=BATCH_SIZE, verbose= 1)

In [ ]:
path4 = TRAINING_DATASET_PATH / "greg_tmp" / "cgan_v0.weights.h5"
model.save_weights(path4)

## TimeGAN

In [ ]:
HIDDEN_DIM, BATCH_SIZE, EPOCHS = 128, 256, 300  # 300 epochs × 3 phases

model = tsgm.models.timeGAN.TimeGAN(
    seq_len=seq_len, n_features=feat_dim, module="gru",
    hidden_dim=HIDDEN_DIM, n_layers=3, batch_size=BATCH_SIZE,
)
model.compile(
    d_optimizer=keras.optimizers.Adam(1e-3),
    g_optimizer=keras.optimizers.Adam(1e-3),
    emb_optimizer=keras.optimizers.Adam(1e-3),
    supgan_optimizer=keras.optimizers.Adam(1e-3),
    ae_optimizer=keras.optimizers.Adam(1e-3),
    emb_loss=keras.losses.MeanSquaredError(),
    clf_loss=keras.losses.BinaryCrossentropy(),
)

# %% train (autoencoder → supervised → joint adversarial)
model.fit(X_scaled, epochs=5, checkpoints_interval=50)

## DdPM

In [ ]:
TIMESTEPS, BATCH_SIZE, EPOCHS = 500, 32, 2000

cls = tsgm.models.zoo["ddpm_denoiser"]
net = cls(seq_len, feat_dim, n_filters=256, n_conv_layers=5).model
ema = cls(seq_len, feat_dim, n_filters=256, n_conv_layers=5).model
model = tsgm.models.ddpm.DDPM(net, ema, timesteps=TIMESTEPS)
model.compile(optimizer=keras.optimizers.Adam(2e-4))

# %% train
model.fit(X_scaled, batch_size=BATCH_SIZE, epochs=5, verbose=1)

In [ ]:
model = tsgm.models.timeGAN.TimeGAN(
    seq_len=168,
    module="gru",
    hidden_dim=64,
    n_features=239,
    n_layers=2,
    batch_size=64,
    gamma=1.0,
)

model.compile()

model.fit(data=X_scaled, epochs=10)

In [ ]:
# we aim at minimizing the discrepancy metric defined in next cell
study = optuna.create_study(direction="minimize")

metric_to_optimize = tsgm.metrics.metrics.DistanceMetric(
            statistics=[
                functools.partial(tsgm.metrics.statistics.axis_max_s, axis=None),
                functools.partial(tsgm.metrics.statistics.axis_min_s, axis=None),
                functools.partial(tsgm.metrics.statistics.axis_max_s, axis=1),
                functools.partial(tsgm.metrics.statistics.axis_min_s, axis=1),
            ],
            discrepancy=lambda x, y: np.linalg.norm(x - y),
        )

# optimizers and the search space for the hyperparameters
def _create_optimizer(trial):
    # optimize the choice of optimizers as well as their parameters
    kwargs = {}
    optimizer_options = ["RMSprop", "Adam", "SGD"]
    optimizer_selected = trial.suggest_categorical("optimizer", optimizer_options)
    if optimizer_selected == "RMSprop":
        kwargs["learning_rate"] = trial.suggest_float(
            "rmsprop_learning_rate", 1e-5, 1e-1, log=True
        )
        kwargs["momentum"] = trial.suggest_float(
            "rmsprop_momentum", 1e-5, 1e-1, log=True
        )
    elif optimizer_selected == "Adam":
        kwargs["learning_rate"] = trial.suggest_float(
            "adam_learning_rate", 1e-5, 1e-1, log=True
        )
    elif optimizer_selected == "SGD":
        kwargs["learning_rate"] = trial.suggest_float(
            "sgd_opt_learning_rate", 1e-5, 1e-1, log=True
        )
        kwargs["momentum"] = trial.suggest_float(
            "sgd_opt_momentum", 1e-5, 1e-1, log=True
        )

    optimizer = getattr(keras.optimizers, optimizer_selected)(**kwargs)
    return optimizer

def objective(trial):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_data = torch.as_tensor(X_scaled, dtype=torch.float32, device=device)

    n_layers = trial.suggest_int("n_layers", 1, 10)
    num_hidden = trial.suggest_int("num_hidden", 4, 128, log=True)

    model = tsgm.models.timeGAN.TimeGAN(
        seq_len=168,
        module="gru",
        hidden_dim=num_hidden,
        n_features=239,
        n_layers=n_layers,
        batch_size=256,
        gamma=1.0,
    ).to(device)

    optimizer = _create_optimizer(trial)
    model.compile(optimizer)

    # if the implementation exposes optimizer state, force it to device
    try:
        for state in optimizer.state.values():
            for k, v in state.items():
                if torch.is_tensor(v):
                    state[k] = v.to(device)
    except Exception:
        pass

    model.fit(data=train_data, epochs=100)

    _y = model.generate(n_samples=10)
    return metric_to_optimize(_y, np.array(train_data[:10].cpu()))

In [ ]:
study.optimize(objective, n_trials=10)